In [1]:
import os
import json
from typing import TypedDict, List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_aws import ChatBedrock, BedrockEmbeddings

In [18]:
aws_model = ChatBedrock(
    model_id = "amazon.nova-micro-v1:0", 
    region_name = "us-east-1", 
    model_kwargs = {
        "temperature": 0.7,
        "max_tokens": 1000
        
    }
)
test_query = "Tell me a joke about FIFA"
response = aws_model.invoke(test_query)
print(response.content)

Sure, here's a light-hearted FIFA joke for you:

Why did the soccer player bring a ladder to the FIFA tournament?

Because he heard the referee always goes "up" when making a high decision!

Hope that brought a smile to your face!


In [6]:
# Agent configurations from Lesson 1
agent_configs = {
    "user_profile_generator": {
        "role": "User Profile Generator",
        "goal": "Analyze user restaurant visit history and social media posts to create a comprehensive profile.",
        "backstory": """You are an expert user behavior analyst with 10 years of experience in the food industry. 
        You excel at identifying patterns in dining behavior and building rich user profiles."""
    },
    "rag_retriever": {
        "role": "RAG Retriever",
        "goal": "Query multimodal vector databases to retrieve relevant restaurants and recipes.",
        "backstory": """You are a data retrieval specialist with expertise in vector databases and semantic search."""
    },
    "food_trend_analyst": {
        "role": "Food Trend Analyst",
        "goal": "Identify current food trends and emerging dining concepts.",
        "backstory": """You are a culinary journalist who has spent 15 years covering food culture across global markets."""
    },
    "food_style_expert": {
        "role": "Food Style Expert",
        "goal": "Analyze cuisine types and flavor profiles to match user preferences.",
        "backstory": """You are a trained chef and culinary anthropologist with expertise in global cuisines."""
    },
    "nutrition_expert": {
        "role": "Nutrition Expert",
        "goal": "Evaluate nutritional content and ensure dietary compliance.",
        "backstory": """You are a registered dietitian with 8 years of clinical experience."""
    },
    "recommendation_expert": {
        "role": "Recommendation Expert",
        "goal": "Synthesize insights from all agents into final recommendations.",
        "backstory": """You are a recommendation systems architect with experience in personalization engines."""
    }
}

print("Agent configurations loaded successfully!")

Agent configurations loaded successfully!


In [7]:
# Define the shared state structure as a dictionary.
# Every node reads from and writes to this state.
INITIAL_STATE = {
    # Input
    "user_input": "",
    
    # Phase 1: User Analysis
    "user_profile": {},
    
    # Phase 2: Data Retrieval
    "retrieved_restaurants": [],
    "retrieved_recipes": [],
    
    # Phase 3: Analysis (Parallel)
    "trend_analysis": {},
    "style_analysis": {},
    "nutrition_analysis": {},
    
    # Phase 4: Synthesis
    "final_recommendations": {},
    
    # Metadata
    "workflow_step": "start"
}

print(f"State structure defined with {len(INITIAL_STATE)} fields:")
for key in INITIAL_STATE:
    print(f"  - {key}")

State structure defined with 9 fields:
  - user_input
  - user_profile
  - retrieved_restaurants
  - retrieved_recipes
  - trend_analysis
  - style_analysis
  - nutrition_analysis
  - final_recommendations
  - workflow_step


Create the node functions
Each node represents an agent's task. A node function:

Receives the current state
Performs the agent's task
Returns an updated state
Let's create node functions for all six agents.


In [19]:
def call_agent(agent_key:str, user_message:str) -> str:   
    """Calls an agent with a specific message and returns its response """
    config = agent_configs[agent_key]
    system_prompt = f""" You are a {config['role']}.
    Your goal :{config['goal']}. 
    Your background : {config['backstory']}
    Respond with a structured,actionable output
    Return ONLY valid JSON with no markdown backticks or extra text. 
    """
    messages = [
        {"role": "system", "content": system_prompt}, 
        {"role": "user", "content": user_message}
    ]
    response = aws_model.invoke(messages)
    return response.content

In [10]:
def node_generate_profile(state:dict)-> dict: 
    """Generates user profile from input data """ 
    print("\n[Phase 1] Generating User profile")
    user_message = f"""Analyze this user data and create a comprehensive profile:

{state['user_input']}

Provide output in JSON format with these keys:
- favorite_cuisines (list)
- dietary_restrictions (list)
- dining_occasions (list)
- price_range (string)
- adventurousness_score (1-10)
- flavor_preferences (list)
- summary (string)
"""
    try: 
        response = call_agent("user_profile_generator", user_message)
        user_profile = json.loads(response)
        print(f"✓ User profile generated: {user_profile.get('summary', 'No summary')}")
    except Exception as e: 
        print(f"⚠ Error generating profile: {e}")
        user_profile = {"error": str(e)}
    
    state["user_profile"] = user_profile
    state["workflow_step"] = "profile_generated"
    return state

In [11]:
def node_retrieve_candidates(state: dict) -> dict:
    """Retrieve restaurant and recipe candidates from vector database."""
    print("\n[Phase 2] Retrieving candidates from vector database...")
    
    profile = state["user_profile"]
    
    user_message = f"""Based on this user profile:
{json.dumps(profile, indent=2)}

Simulate retrieving top 20 restaurants and top 20 recipes from a vector database.

Return JSON with two arrays:
- restaurants: [{{"name": str, "cuisine": str, "price": str, "rating": float, "description": str}}]
- recipes: [{{"name": str, "cuisine": str, "difficulty": str, "prep_time": str, "description": str}}]

Make the results realistic and diverse.
"""
    
    try:
        response = call_agent("rag_retriever", user_message)
        retrieved_data = json.loads(response)
        restaurants = retrieved_data.get("restaurants", [])
        recipes = retrieved_data.get("recipes", [])
        print(f"✓ Retrieved {len(restaurants)} restaurants and {len(recipes)} recipes")
    except Exception as e:
        print(f"⚠ Error retrieving candidates: {e}")
        restaurants, recipes = [], []
    
    state["retrieved_restaurants"] = restaurants
    state["retrieved_recipes"] = recipes
    state["workflow_step"] = "candidates_retrieved"
    return state

Node 3: Analyze trends

In [12]:
def node_analyze_trends(state: dict) -> dict:
    """Analyze food trends in the retrieved candidates."""
    print("\n[Phase 3a] Analyzing food trends...")
    
    restaurants = state["retrieved_restaurants"]
    recipes = state["retrieved_recipes"]
    
    user_message = f"""Analyze current food trends in these options:

Restaurants: {json.dumps(restaurants[:5], indent=2)}
Recipes: {json.dumps(recipes[:5], indent=2)}

Identify 3-5 relevant trends and explain how they align with modern dining culture.
Return JSON: {{"trends": [{{"name": str, "description": str, "relevance": str}}]}}
"""
    
    try:
        response = call_agent("food_trend_analyst", user_message)
        trend_analysis = json.loads(response)
        print(f"✓ Identified {len(trend_analysis.get('trends', []))} trends")
    except Exception as e:
        print(f"⚠ Error analyzing trends: {e}")
        trend_analysis = {"error": str(e)}
    
    state["trend_analysis"] = trend_analysis
    return state


In [21]:
def node_analyze_styles(state: dict) -> dict:
    """Analyze food styles and flavor profiles."""
    print("\n[Phase 3b] Analyzing food styles...")
    
    restaurants = state["retrieved_restaurants"]
    recipes = state["retrieved_recipes"]
    profile = state["user_profile"]
    
    user_message = f"""Analyze the food styles and flavor profiles of these options for a user with this profile:
Profile: {json.dumps(profile, indent=2)}
Restaurants: {json.dumps(restaurants[:5], indent=2)}
Recipes: {json.dumps(recipes[:5], indent=2)}
Identify key flavor profiles, cooking styles, and cuisine characteristics.
Return ONLY valid JSON no markdown backticks: {{"style_analysis": [{{"cuisine": str, "flavor_profile": str, "cooking_style": str, "match_score": float}}]}}"""
    
    try:
        response = call_agent("food_style_expert", user_message)
        style_analysis = json.loads(response)
        print(f"✓ Style analysis completed")
    except Exception as e:
        print(f"⚠ Error analyzing styles: {e}")
        style_analysis = {"error": str(e)}
    
    state["style_analysis"] = style_analysis
    return state

print("✅ node_analyze_styles ready")

✅ node_analyze_styles ready


Node 5: Evaluate nutrition

In [14]:
def node_evaluate_nutrition(state: dict) -> dict:
    """Evaluate nutritional aspects and dietary compliance."""
    print("\n[Phase 3c] Evaluating nutrition...")
    
    restaurants = state["retrieved_restaurants"]
    recipes = state["retrieved_recipes"]
    profile = state["user_profile"]
    
    user_message = f"""Evaluate the nutritional fit of these options:

User Profile: {json.dumps(profile, indent=2)}
Restaurants: {json.dumps(restaurants[:5], indent=2)}
Recipes: {json.dumps(recipes[:5], indent=2)}

Check dietary restrictions, allergens, and nutritional balance.
Return JSON: {{"compliant_items": [], "flagged_items": [], "nutritional_highlights": []}}
"""
    
    try:
        response = call_agent("nutrition_expert", user_message)
        nutrition_analysis = json.loads(response)
        print(f"✓ Nutrition evaluation completed")
    except Exception as e:
        print(f"⚠ Error evaluating nutrition: {e}")
        nutrition_analysis = {"error": str(e)}
    
    state["nutrition_analysis"] = nutrition_analysis
    return state

Node 6: Generate recommendations

In [15]:
def node_generate_recommendations(state: dict) -> dict:
    """Synthesize all analyses into final recommendations."""
    print("\n[Phase 4] Generating final recommendations...")
    
    user_message = f"""Synthesize these insights into top 5 restaurant and top 5 recipe recommendations:

User Profile: {json.dumps(state['user_profile'], indent=2)}
Restaurants: {json.dumps(state['retrieved_restaurants'][:10], indent=2)}
Recipes: {json.dumps(state['retrieved_recipes'][:10], indent=2)}
Trends: {json.dumps(state['trend_analysis'], indent=2)}
Styles: {json.dumps(state['style_analysis'], indent=2)}
Nutrition: {json.dumps(state['nutrition_analysis'], indent=2)}

Return JSON:
{{
  "restaurants": [{{"name": str, "reasoning": str}}],
  "recipes": [{{"name": str, "reasoning": str}}]
}}

Each reasoning should be 2-3 sentences explaining why it's a great match.
"""
    
    try:
        response = call_agent("recommendation_expert", user_message)
        recommendations = json.loads(response)
        print(f"✓ Generated {len(recommendations.get('restaurants', []))} restaurant recommendations")
        print(f"✓ Generated {len(recommendations.get('recipes', []))} recipe recommendations")
    except Exception as e:
        print(f"⚠ Error generating recommendations: {e}")
        recommendations = {"error": str(e)}
    
    state["final_recommendations"] = recommendations
    state["workflow_step"] = "complete"
    return state

In [16]:
def run_workflow(user_input: str) -> dict:
    """Run the full multi-agent workflow.
    
    Phases:
      1. User Analysis      (sequential)
      2. Data Retrieval      (sequential)
      3. Analysis            (parallel – trends, styles, nutrition)
      4. Synthesis           (sequential)
    """
    
    # Initialize shared state
    state = {
        "user_input": user_input,
        "user_profile": {},
        "retrieved_restaurants": [],
        "retrieved_recipes": [],
        "trend_analysis": {},
        "style_analysis": {},
        "nutrition_analysis": {},
        "final_recommendations": {},
        "workflow_step": "start"
    }
    
    # Phase 1 – Sequential
    state = node_generate_profile(state)
    
    # Phase 2 – Sequential
    state = node_retrieve_candidates(state)
    
    # Phase 3 – Parallel using ThreadPoolExecutor
    print("\n[Phase 3] Running analysis agents in parallel...")
    
    # Each function needs its own copy of state to read from,
    # and we merge their outputs back afterwards.
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_trends   = executor.submit(node_analyze_trends, dict(state))
        future_styles   = executor.submit(node_analyze_styles, dict(state))
        future_nutrition = executor.submit(node_evaluate_nutrition, dict(state))
        
        result_trends   = future_trends.result()
        result_styles   = future_styles.result()
        result_nutrition = future_nutrition.result()
        
    # Merge parallel results back into state
    state["trend_analysis"]    = result_trends["trend_analysis"]
    state["style_analysis"]    = result_styles["style_analysis"]
    state["nutrition_analysis"] = result_nutrition["nutrition_analysis"]
    
    # Phase 4 – Sequential
    state = node_generate_recommendations(state)
    
    return state

print("✓ Workflow function built successfully!")

✓ Workflow function built successfully!


Test Case 1: The health-conscious user

In [22]:
test_user_1 = """
Restaurant Visit History:
- Visited "Green Bowl" (Vegan, $$) 8 times
- Visited "Mediterranean Grill" (Mediterranean, $$) 5 times
- Visited "Juice Lab" (Smoothies, $) 3 times

Social Media Posts:
- "Loving my plant-based journey! 🌱"
- "This gluten-free Mediterranean bowl is amazing!"
- "Fresh juice is the best way to start the day."

Dietary Restrictions: Vegan, Gluten-Free
"""

print("="*80)
print("TEST CASE 1: Health-Conscious User")
print("="*80)

try:
    result_1 = run_workflow(test_user_1)
    print("\n" + "="*80)
    print("FINAL RECOMMENDATIONS")
    print("="*80)
    print(json.dumps(result_1["final_recommendations"], indent=2))
except Exception as e:
    print(f"\nTest requires valid aws credentials. Error: {e}")

TEST CASE 1: Health-Conscious User

[Phase 1] Generating User profile
✓ User profile generated: This user has a strong preference for vegan and Mediterranean cuisines, frequently visiting places that align with their dietary restrictions. They enjoy dining out for casual and health-conscious occasions, showing a good balance between healthy eating and flavor preference.

[Phase 2] Retrieving candidates from vector database...
⚠ Error retrieving candidates: Unterminated string starting at: line 22 column 101 (char 3476)

[Phase 3] Running analysis agents in parallel...

[Phase 3a] Analyzing food trends...

[Phase 3b] Analyzing food styles...

[Phase 3c] Evaluating nutrition...
✓ Style analysis completed
✓ Nutrition evaluation completed
✓ Identified 5 trends

[Phase 4] Generating final recommendations...
✓ Generated 5 restaurant recommendations
✓ Generated 5 recipe recommendations

FINAL RECOMMENDATIONS
{
  "restaurants": [
    {
      "name": "Green Bowl",
      "reasoning": "Green Bowl

Test Case 2: The adventurous foodie

In [23]:
test_user_2 = """
Restaurant Visit History:
- Visited "Omakase Sushi" (Japanese Fine Dining, $$$$) 4 times
- Visited "Street Food Market" (International Fusion, $$) 6 times
- Visited "Molecular Gastronomy Lab" (Experimental, $$$$) 2 times

Social Media Posts:
- "Mind-blown by the 12-course tasting menu! 🤯"
- "Trying crickets for the first time. Surprisingly good!"
- "This molecular take on traditional ramen is art."

Dietary Restrictions: None
"""

print("="*80)
print("TEST CASE 2: Adventurous Foodie")
print("="*80)

try:
    result_2 = run_workflow(test_user_2)
    print("\n" + "="*80)
    print("FINAL RECOMMENDATIONS")
    print("="*80)
    print(json.dumps(result_2["final_recommendations"], indent=2))
except Exception as e:
    print(f"\nTest requires valid aws credentials. Error: {e}")

TEST CASE 2: Adventurous Foodie

[Phase 1] Generating User profile
✓ User profile generated: The user has a high affinity for Japanese fine dining and international fusion cuisines. They enjoy high-end, sophisticated dining experiences and are not averse to trying unconventional and experimental dishes. They have a penchant for adventurous and unique flavor profiles.

[Phase 2] Retrieving candidates from vector database...
⚠ Error retrieving candidates: Expecting value: line 120 column 21 (char 3992)

[Phase 3] Running analysis agents in parallel...

[Phase 3a] Analyzing food trends...

[Phase 3b] Analyzing food styles...

[Phase 3c] Evaluating nutrition...
✓ Nutrition evaluation completed✓ Style analysis completed

✓ Identified 4 trends

[Phase 4] Generating final recommendations...
✓ Generated 5 restaurant recommendations
✓ Generated 5 recipe recommendations

FINAL RECOMMENDATIONS
{
  "restaurants": [
    {
      "name": "Sushi Zanmai",
      "reasoning": "Sushi Zanmai offers an inno

In [26]:
def evaluate_recommendations(result: Dict[str, Any]):
    """Evaluate the quality of recommendations."""
    print("\n" + "="*80)
    print("RECOMMENDATION EVALUATION")
    print("="*80)
    
    profile = result.get("user_profile", {})
    recommendations = result.get("final_recommendations", {})
    
    # Check if recommendations exist
    restaurants = recommendations.get("restaurants", [])
    recipes = recommendations.get("recipes", [])
    
    print(f"\n✓ Number of restaurant recommendations: {len(restaurants)}")
    print(f"✓ Number of recipe recommendations: {len(recipes)}")
    
    # Check dietary compliance
    dietary_restrictions = profile.get("dietary_restrictions", [])
    if dietary_restrictions:
        print(f"\n✓ Dietary restrictions identified: {', '.join(dietary_restrictions)}")
        print("  → Check if recommendations respect these restrictions")
    
    # Check diversity
    favorite_cuisines = profile.get("favorite_cuisines", [])
    if favorite_cuisines:
        print(f"\n✓ Favorite cuisines: {', '.join(favorite_cuisines)}")
        print("  → Check if recommendations include these cuisines")
    
    # Evaluate reasoning quality
    if restaurants:
        print(f"\n✓ First restaurant recommendation:")
        print(f"  Name: {restaurants[0].get('name', 'N/A')}")
        print(f"  Reasoning: {restaurants[0].get('reasoning', 'N/A')}")
    
    if recipes:
        print(f"\n✓ First recipe recommendation:")
        print(f"  Name: {recipes[0].get('name', 'N/A')}")
        print(f"  Reasoning: {recipes[0].get('reasoning', 'N/A')}")
    
    print("\n" + "="*80)

In [25]:
# Evaluate Test Case 1 if available
try:
    if 'result_1' in locals():
        evaluate_recommendations(result_1)
except Exception as e:
    print(f"Evaluation requires completed test run: {e}")


RECOMMENDATION EVALUATION

✓ Number of restaurant recommendations: 5
✓ Number of recipe recommendations: 5

✓ Dietary restrictions identified: Vegan, Gluten-Free
  → Check if recommendations respect these restrictions

✓ Favorite cuisines: Vegan, Mediterranean
  → Check if recommendations include these cuisines

✓ First restaurant recommendation:
  Name: Green Bowl
  Reasoning: Green Bowl offers a minimalist, fresh vegan Mediterranean menu that aligns with the user's flavor preferences for healthy, flavorful cuisine. The restaurant's commitment to sustainability and waste reduction also matches the user's values.

✓ First recipe recommendation:
  Name: Quinoa Salad with Lemon and Herbs
  Reasoning: This recipe provides a fresh, gluten-free, and vegan option that is packed with protein from quinoa and healthy fats from olive oil, aligning perfectly with the user's nutritional needs and Mediterranean flavor preferences.

